# agentic-civic-resolution-app
Agent 1: Complaint Classification Agent

**Role:** Raw complaint text → structured `category`, `subcategory`, `confidence`

Uses Databricks Foundation Model API (Llama 3.1 70B) with structured JSON output.

**COMMAND** 

%pip install databricks-langchain mlflow pydantic --quiet

dbutils.library.restartPython()

In [0]:
%pip install databricks-langchain mlflow pydantic --quiet

#dbutils.library.restartPython()

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS civicops.agents")

## 0. Config & Imports

In [0]:
import json, re, mlflow, requests
from pydantic import BaseModel, Field
from databricks_langchain import ChatDatabricks

In [0]:
# ── MLflow Experiment Setup ───────────────────────────────────────────────────
# Databricks MLflow requires every parent folder to exist before set_experiment().
# We create them via the Workspace REST API (mkdirs), which handles nested paths
# atomically and is idempotent (safe to re-run).

def ensure_mlflow_experiment(experiment_path: str) -> str:
    """
    1. Creates every parent folder in experiment_path via Workspace mkdirs API.
    2. Calls mlflow.set_experiment() — creates experiment if it doesn't exist.
    Returns the experiment_id.
    """
    ctx     = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host    = ctx.apiUrl().get()
    token   = ctx.apiToken().get()
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    # Build every parent prefix: /civicops  then  /civicops/agents
    parts = experiment_path.strip("/").split("/")
    for depth in range(1, len(parts)):          # stop before the leaf (experiment name)
        folder = "/" + "/".join(parts[:depth])
        resp   = requests.post(
            f"{host}/api/2.0/workspace/mkdirs",
            headers=headers,
            json={"path": folder},
        )
        body = resp.json()
        if resp.status_code == 200:
            print(f"  ✓ folder: {folder}")
        elif body.get("error_code") == "RESOURCE_ALREADY_EXISTS":
            print(f"  ✓ exists: {folder}")
        else:
            raise RuntimeError(f"mkdirs failed for {folder}: {body}")

    exp = mlflow.set_experiment(experiment_path)
    return exp.experiment_id


EXPERIMENT_ID = ensure_mlflow_experiment("/civicops/agents/complaint_classifier")
print(f"✓ MLflow experiment ready (id={EXPERIMENT_ID})")



In [0]:
# ── Auto-discover a working Foundation Model endpoint ────────────────────────
# Lists all serving endpoints in your workspace and picks the first available
# chat/instruct model. Override LLM_ENDPOINT manually if you want a specific one.

def discover_llm_endpoint() -> str:
    ctx     = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host    = ctx.apiUrl().get()
    token   = ctx.apiToken().get()
    headers = {"Authorization": f"Bearer {token}"}

    resp = requests.get(f"{host}/api/2.0/serving-endpoints", headers=headers)
    resp.raise_for_status()
    endpoints = resp.json().get("endpoints", [])

    # Priority order: prefer llama > dbrx > mpt > any ready endpoint
    priority_keywords = ["llama", "dbrx", "mixtral", "mpt"]
    ready = [e for e in endpoints if e.get("state", {}).get("ready") == "READY"]

    if not ready:
        raise RuntimeError(
            "No READY serving endpoints found in your workspace.\n"
            "Go to Serving → Create a Foundation Model endpoint first."
        )

    print("Available READY endpoints:")
    for e in ready:
        print(f"  • {e['name']}")

    for kw in priority_keywords:
        for e in ready:
            if kw in e["name"].lower():
                print(f"\n✓ Auto-selected endpoint: {e['name']}")
                return e["name"]

    # Fallback: just take the first ready one
    chosen = ready[0]["name"]
    print(f"\n✓ Auto-selected endpoint (first ready): {chosen}")
    return chosen

In [0]:
# To pin a specific endpoint, replace the line below with e.g.:
#   LLM_ENDPOINT = "databricks-dbrx-instruct"
LLM_ENDPOINT = discover_llm_endpoint()

## 1. Output Schema (Pydantic)

In [0]:
class ClassificationOutput(BaseModel):
    category:    str   = Field(description="Top-level complaint category")
    subcategory: str   = Field(description="Specific sub-type within category")
    confidence:  float = Field(ge=0.0, le=1.0, description="Model confidence 0–1")
    reasoning:   str   = Field(description="One sentence explaining the classification")

## 2. System Prompt

In [0]:
SYSTEM_PROMPT = """
You are the Complaint Classification Agent for CivicOps AI.

Classify the citizen complaint into ONE category and ONE subcategory from this table:

| Category           | Valid Subcategories                                                         |
|--------------------|----------------------------------------------------------------------------|
| Sanitation         | Garbage Overflow, Illegal Dumping, Dead Animal, Littering, Clogged Drain   |
| Infrastructure     | Pothole, Road Crack, Sidewalk Damage, Bridge Damage, Building Risk         |
| Noise              | Residential Noise, Commercial Noise, Construction Noise, Vehicle Noise     |
| Water & Sewage     | Water Leakage, No Water Supply, Sewage Overflow, Water Quality Issue       |
| Electricity        | Streetlight Outage, Power Outage, Exposed Wires, Transformer Issue         |
| Road & Traffic     | Traffic Signal Fault, Blocked Road, Illegal Parking, Missing Road Markings |
| Public Safety      | Stray Animals, Fire Hazard, Fallen Tree, Suspicious Activity               |
| Parks & Environment| Overgrown Vegetation, Park Damage, Water Body Pollution, Deforestation     |

Confidence guide: 0.9+ = unambiguous | 0.6-0.89 = likely | <0.6 = ambiguous.

Respond ONLY with valid JSON - no markdown fences, no extra text:
{
  "category": "<string>",
  "subcategory": "<string>",
  "confidence": <float>,
  "reasoning": "<one sentence>"
}
""".strip()

## 3. Agent Class

In [0]:
class ComplaintClassifierAgent:
    """Stateless LLM-backed classification agent."""

    def __init__(self, endpoint: str = LLM_ENDPOINT):
        self.llm      = ChatDatabricks(endpoint=endpoint, temperature=0.0)
        self.endpoint = endpoint

    def classify(self, complaint_text: str) -> ClassificationOutput:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Complaint: {complaint_text.strip()}"},
        ]
        with mlflow.start_run(run_name="classify", nested=True):
            mlflow.log_params({"endpoint": self.endpoint,
                               "input":    complaint_text[:200]})
            response = self.llm.invoke(messages)
            result   = self._parse(response.content)
            mlflow.log_metric("confidence", result.confidence)
        return result

    def classify_batch(self, texts: list[str]) -> list[ClassificationOutput]:
        return [self.classify(t) for t in texts]

    @staticmethod
    def _parse(raw: str) -> ClassificationOutput:
        raw = raw.strip()
        try:
            return ClassificationOutput(**json.loads(raw))
        except Exception:
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            if m:
                return ClassificationOutput(**json.loads(m.group()))
            raise ValueError(f"Unparseable classifier output:\n{raw}")


classifier_agent = ComplaintClassifierAgent()
print("✓ Classifier agent ready.")

## 4. Smoke Test

In [0]:
_tests = [
    "Garbage overflow near Whitefield",
    "Streetlight on MG Road has been out for 3 days",
    "Massive pothole near the school on 5th Ave",
    "Loud music from the bar every night after midnight",
    "Water pipeline burst on Main Street — road flooded",
]

print(f"{'Complaint':<52} {'Category':<22} {'Sub-category':<26} Conf")
print("-" * 108)
for t in _tests:
    r = classifier_agent.classify(t)
    print(f"{t[:51]:<52} {r.category:<22} {r.subcategory:<26} {r.confidence:.2f}")

## 5. Register in MLflow Model Registry

In [0]:

import pandas as pd
from mlflow.models.signature import infer_signature

class _ClassifierWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.agent = ComplaintClassifierAgent()

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        return pd.DataFrame(
            [self.agent.classify(str(t)).model_dump()
             for t in model_input["complaint_text"]]
        )


In [0]:

# Build signature from a concrete example
_input_example  = pd.DataFrame({"complaint_text": ["Garbage overflow near Whitefield"]})
_output_example = pd.DataFrame([ClassificationOutput(
    category="Sanitation",
    subcategory="Garbage Overflow",
    confidence=0.95,
    reasoning="Keywords indicate a sanitation issue.",
).model_dump()])
_signature = infer_signature(_input_example, _output_example)

spark.sql("CREATE CATALOG IF NOT EXISTS civicops")
spark.sql("CREATE SCHEMA IF NOT EXISTS civicops.agents")
print("✓ civicops.agents schema ready.")



In [0]:
with mlflow.start_run(run_name="register_classifier"):
    mlflow.pyfunc.log_model(
        name="complaint_classifier",          # replaces deprecated artifact_path
        python_model=_ClassifierWrapper(),
        pip_requirements=["databricks-langchain", "pydantic"],
        input_example=_input_example,
        signature=_signature,
        registered_model_name="civicops.agents.complaint_classifier",
    )
print("✓ Classifier registered in MLflow Model Registry.")
